# gpt-5.6-luna — KV(프롬프트) 캐시 벤치마크

**대상 모델**: `gpt-5.6-luna` (GPT 5.6 세대)
**측정 항목**: 캐시 MISS(cold) vs HIT(warm) 의 **TTFT**, **전체 latency**, **cached / cache_write 토큰**, **비용**

## ⚠️ GPT 5.6 세대는 캐싱 회계가 바뀌었습니다

5.6 부터 OpenAI 도 **캐시 쓰기와 읽기를 분리**해 리포트하고, **쓰기에 1.25배를 과금**합니다.
Anthropic 의 `cache_creation` / `cache_read` 2단 회계와 같은 구조가 OpenAI 에도 생긴 것입니다.

실측 usage 덤프:
```
gpt-5.6-luna  1회차(콜드): cache_write_tokens=6010, cached_tokens=0
              2회차(웜):   cache_write_tokens=0,    cached_tokens=6010
gpt-5.5       cache_write_tokens=None  (항상)
gpt-5.4-mini  cache_write_tokens=None  (항상)
```

| | GPT 5.6 이상 | GPT 5.5 이하 |
|---|---|---|
| `cache_write_tokens` | **실제 값** | 항상 `None`/`0` |
| 쓰기 과금 | **1.25배** | 무료 |
| 캐시 입자도 | 거의 전량 (6,010/6,013) | 128토큰 블록 절삭 (5,888) |
| TTL 제어 | `prompt_cache_options.ttl="30m"` (유일값·기본값, 최소 30분 보장) | `prompt_cache_retention` (5~10분, 확장 24h) |

**실무 함의**: 5.6 부터는 **재사용 없는 1회성 요청이 오히려 1.25배 비쌉니다.**
쓰기 1.25배를 읽기 90% 할인으로 회수하려면 같은 프리픽스를 **약 1.3회 이상** 재사용해야 합니다
— 5.5 이하엔 없던 리스크입니다. 아래 '쓰기/읽기 손익' 셀에서 직접 계산합니다.

## 실험 설계
- **MISS 강제**: 매 호출마다 system 프리픽스 맨 앞에 유니크 nonce 삽입 → 프리픽스가 매번 달라져 항상 미스.
  이때 매번 **캐시를 새로 쓰므로**(`cache_write_tokens` 발생) 5.6 에서는 MISS 가 곧 "쓰기 비용을 내는 상태"입니다.
- **HIT 유도**: 완전히 동일한 컨텍스트로 워밍업한 뒤 반복 호출(+ 동일 `prompt_cache_key`).
- 출력은 짧게 고정하고 `reasoning_effort="none"` → prefill 캐싱 효과에 집중.

> ⚠️ 캐시는 최소 30분 유지되지만 서버 사정에 따라 evict 될 수 있으니 MISS→HIT 를 **연속 실행**하세요.


In [ ]:
# === 셋업 & 실험 설정 ===
import json, os, pathlib, statistics, time, uuid

from openai import OpenAI

try:
    from dotenv import load_dotenv
    load_dotenv()  # 프로젝트 루트 .env 의 OPENAI_API_KEY 로드
except Exception:
    pass

client = OpenAI()

# ------- 조절 가능한 설정 -------
MODEL             = "gpt-5.6-luna"
TARGET_TOKENS     = 20_000              # 컨텍스트 목표 크기 (200K 로 올리려면 여기만 변경)
MAX_OUTPUT_TOKENS = 32                  # 출력 짧게 고정 → prefill 캐싱 효과에 집중
REASONING_EFFORT  = "none"              # 지원값 실측: none / low / medium / high (minimal 은 400)
N_REPEAT          = 3                   # MISS / HIT 각각 반복 횟수
N_WARMUP          = 2                   # 캐시 적재용 워밍업 호출 횟수
CACHE_KEY         = "kv-gpt56-luna"     # HIT 호출 라우팅 고정용 prompt_cache_key

# ------- 단가 ($ / 1M tokens) — 공식 pricing 확인값 -------
PRICE_INPUT        = 1.00    # uncached 입력
PRICE_CACHED_INPUT = 0.10    # cached 입력 (90% 할인)
PRICE_OUTPUT       = 6.00    # 출력
CACHE_WRITE_MULT   = 1.25    # ⚠️ 5.6 세대 고유 — 캐시 쓰기는 입력가의 1.25배 (5.5 이하는 무료)


def calc_cost(prompt_tokens, cached, write, completion_tokens):
    """5.6 세대는 입력을 3갈래로 나눠 과금: uncached(1x) / write(1.25x) / cached(0.1x)."""
    uncached = (prompt_tokens or 0) - (cached or 0) - (write or 0)
    return (uncached * PRICE_INPUT
            + (write or 0) * PRICE_INPUT * CACHE_WRITE_MULT
            + (cached or 0) * PRICE_CACHED_INPUT
            + (completion_tokens or 0) * PRICE_OUTPUT) / 1_000_000


print(f"모델: {MODEL} | 목표 토큰: {TARGET_TOKENS:,} | 반복: {N_REPEAT} | 출력한도: {MAX_OUTPUT_TOKENS}")
print(f"단가($/1M): in {PRICE_INPUT} | cached-in {PRICE_CACHED_INPUT} | out {PRICE_OUTPUT} "
      f"| write {PRICE_INPUT * CACHE_WRITE_MULT} ({CACHE_WRITE_MULT}x)")

In [ ]:
# === 토큰 카운터 (tiktoken 있으면 정밀, 없으면 근사) ===
# 근사를 쓰더라도 실제 토큰 수는 아래에서 API usage 로 실측 보정됩니다.
try:
    import tiktoken
    try:
        _enc = tiktoken.encoding_for_model(MODEL)
    except KeyError:
        _enc = tiktoken.get_encoding("o200k_base")

    def count_tokens(s: str) -> int:
        return len(_enc.encode(s))

    HAVE_TIKTOKEN = True
except Exception:
    def count_tokens(s: str) -> int:
        return max(1, round(len(s) / 3.7))

    HAVE_TIKTOKEN = False

print("tiktoken 정밀 카운트:", HAVE_TIKTOKEN)

In [ ]:
# === 합성 멀티턴 컨텍스트 생성 ===
SYSTEM_BASE = (
    "You are a senior software engineer helping with a long, ongoing technical "
    "design discussion about distributed systems. Keep the final answer very brief."
)

_Q = (
    "Question {i}: In our distributed order-processing service we occasionally see "
    "duplicate side effects when a consumer redelivers a message. Walk me through how "
    "idempotency keys, an inbox table, and exactly-once semantics interact, and where "
    "the usual mistakes are (batch {i})."
)
_A = (
    "Answer {i}: Redelivery is expected under at-least-once delivery, so the consumer "
    "must be idempotent. Derive an idempotency key from the message id and insert it into "
    "an inbox table inside the SAME transaction that applies the side effect; on redelivery "
    "the unique constraint short-circuits the duplicate. True exactly-once across a network "
    "is impossible, so you approximate it with at-least-once delivery plus idempotent effects. "
    "Common mistakes: committing the inbox row in a separate transaction from the effect, "
    "using a non-deterministic key, and forgetting that outbound publishes also need the "
    "outbox pattern to avoid dual-write races (batch {i})."
)


def build_conversation(target_tokens: int):
    messages = [{"role": "system", "content": SYSTEM_BASE}]
    total = count_tokens(SYSTEM_BASE)
    i = 0
    while total < target_tokens - 200:          # 최종 질문 몫으로 여유
        u, a = _Q.format(i=i), _A.format(i=i)
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
        total += count_tokens(u) + count_tokens(a)
        i += 1
    # 이 최종 질문도 프리픽스에 포함되므로 매 호출 동일해야 히트됨
    messages.append({
        "role": "user",
        "content": "Summarize the single most important rule from the discussion above in one sentence.",
    })
    return messages, total


CONV, approx_tokens = build_conversation(TARGET_TOKENS)
print(f"메시지 {len(CONV)}개 | 근사 토큰 {approx_tokens:,} (실측은 첫 호출 usage 로 확인)")

In [ ]:
# === 측정 유틸: 스트리밍으로 TTFT + latency + cached/write 토큰 ===
def timed_call(messages, *, unique: bool):
    """unique=True → 매번 유니크 프리픽스로 캐시 MISS 강제 / False → 동일 프리픽스로 HIT 유도."""
    msgs = messages
    nonce = None
    if unique:
        nonce = uuid.uuid4().hex
        head = messages[0]
        # system 프리픽스 맨 앞에 nonce 삽입 → 프리픽스 전체가 매번 달라짐
        msgs = [{**head, "content": f"[nonce-{nonce}] " + head["content"]}] + messages[1:]

    kwargs = dict(
        model=MODEL,
        messages=msgs,
        max_completion_tokens=MAX_OUTPUT_TOKENS,
        stream=True,
        stream_options={"include_usage": True},
        prompt_cache_key=("miss-" + nonce) if unique else CACHE_KEY,
    )
    if REASONING_EFFORT:
        kwargs["reasoning_effort"] = REASONING_EFFORT

    t0 = time.perf_counter()
    ttft, usage = None, None
    for chunk in client.chat.completions.create(**kwargs):
        if ttft is None and chunk.choices:      # 첫 스트림 이벤트 = prefill 종료 근사
            ttft = time.perf_counter() - t0
        u = getattr(chunk, "usage", None)
        if u is not None:
            usage = u
    total = time.perf_counter() - t0
    if ttft is None:
        ttft = total

    cached = write = 0
    ptoks = ctoks = None
    if usage is not None:
        ptoks = usage.prompt_tokens
        ctoks = getattr(usage, "completion_tokens", None)
        ptd = getattr(usage, "prompt_tokens_details", None)
        if ptd is not None:
            cached = getattr(ptd, "cached_tokens", 0) or 0
            # ⚠️ 5.6 세대 고유 필드. 5.5 이하에서는 항상 None 이라 0 으로 떨어집니다.
            write = getattr(ptd, "cache_write_tokens", None) or 0
    return {"ttft": ttft, "total": total, "cached": cached, "write": write,
            "prompt_tokens": ptoks, "completion_tokens": ctoks,
            "cost": calc_cost(ptoks, cached, write, ctoks)}

In [ ]:
# === 워밍업: 캐시 적재 + 실제 prompt_tokens 실측 ===
print("워밍업(캐시 적재) 중...")
first = timed_call(CONV, unique=False)
print(f"  1차: prompt_tokens={first['prompt_tokens']:,} | cached={first['cached']:,} | write={first['write']:,} "
      f"| TTFT={first['ttft'] * 1000:.0f}ms total={first['total'] * 1000:.0f}ms | cost=${first['cost']:.5f}")
print(f"  → 실측 입력 토큰 {first['prompt_tokens']:,} (목표 {TARGET_TOKENS:,})")
if first["write"]:
    print(f"  → 1차 호출에서 cache_write {first['write']:,} 토큰 발생 (1.25배 과금) — 5.6 세대의 특징입니다.")

for w in range(max(0, N_WARMUP - 1)):
    r = timed_call(CONV, unique=False)
    print(f"  워밍업 {w + 2}: cached={r['cached']:,} write={r['write']:,} "
          f"TTFT={r['ttft'] * 1000:.0f}ms total={r['total'] * 1000:.0f}ms")

In [ ]:
# === 벤치마크: MISS(cold) vs HIT(warm) ===
def bench(label, unique, n):
    rows = []
    for k in range(n):
        r = timed_call(CONV, unique=unique)
        rows.append(r)
        print(f"  [{label}] {k + 1}/{n}: TTFT={r['ttft'] * 1000:7.0f}ms  total={r['total'] * 1000:7.0f}ms  "
              f"cached={r['cached']:>7,}  write={r['write']:>7,}  cost=${r['cost']:.5f}")
    return rows


print("=== 캐시 MISS (cold, 매 호출 유니크 프리픽스 → 매번 캐시를 새로 씀) ===")
miss = bench("MISS", unique=True, n=N_REPEAT)
print("=== 캐시 HIT (warm, 동일 프리픽스 반복) ===")
hit = bench("HIT", unique=False, n=N_REPEAT)

In [ ]:
# === 결과 집계 & 개선율 ===
def stat(rows, key):
    xs = [r[key] for r in rows]
    return {"mean": statistics.mean(xs), "p50": statistics.median(xs), "min": min(xs)}


def ms(x):
    return f"{x * 1000:.0f}"


ptoks = first["prompt_tokens"] or approx_tokens
print(f"모델 {MODEL} | 반복 {N_REPEAT}회 | 입력 {ptoks:,} tok\n")

summary = {}
for name, key in [("TTFT (첫 토큰)", "ttft"), ("전체 latency", "total")]:
    m, h = stat(miss, key), stat(hit, key)
    imp = (1 - h["mean"] / m["mean"]) * 100 if m["mean"] else 0
    speedup = (m["mean"] / h["mean"]) if h["mean"] else float("inf")
    summary[key] = {"miss": m, "hit": h, "improve_pct": imp, "speedup": speedup}
    print(f"[{name}]")
    print(f"  MISS  mean {ms(m['mean'])}ms  p50 {ms(m['p50'])}ms  min {ms(m['min'])}ms")
    print(f"  HIT   mean {ms(h['mean'])}ms  p50 {ms(h['p50'])}ms  min {ms(h['min'])}ms")
    print(f"  -> 개선: {imp:5.1f}% 단축  ({speedup:.2f}x 빠름)\n")

avg_cached_hit = statistics.mean([r["cached"] for r in hit])
avg_cached_miss = statistics.mean([r["cached"] for r in miss])
avg_write_hit = statistics.mean([r["write"] for r in hit])
avg_write_miss = statistics.mean([r["write"] for r in miss])
print(f"cached_tokens 평균 — HIT {avg_cached_hit:,.0f} / MISS {avg_cached_miss:,.0f} (전체 {ptoks:,})")
print(f"cache_write_tokens 평균 — HIT {avg_write_hit:,.0f} / MISS {avg_write_miss:,.0f}")
print(f"HIT 캐시 적중률 ≈ {avg_cached_hit / ptoks:.1%}")
if avg_cached_hit < 1024:
    print("⚠️ HIT cached_tokens 가 1024 미만입니다 — 캐시 미적재/evict 가능. MISS→HIT 를 연속 재실행하세요.")

miss_cost = statistics.mean([r["cost"] for r in miss])
hit_cost = statistics.mean([r["cost"] for r in hit])
save_pct = (1 - hit_cost / miss_cost) * 100 if miss_cost else 0
run_total = first["cost"] + sum(r["cost"] for r in miss) + sum(r["cost"] for r in hit)
print(f"\n[비용] 호출당 평균 — MISS ${miss_cost:.5f} / HIT ${hit_cost:.5f} → {save_pct:.1f}% 절감")
print(f"  이번 벤치 총비용 ≈ ${run_total:.4f}")

In [ ]:
# === 쓰기/읽기 손익 — 5.6 세대에서만 의미 있는 계산 ===
# 쓰기 1.25배를 읽기 90% 할인으로 회수하려면 같은 프리픽스를 몇 번 써야 하는가?
if avg_write_miss == 0 and avg_write_hit == 0:
    print("cache_write_tokens 가 0 입니다 — 5.5 이하 모델이거나 캐시 쓰기가 발생하지 않았습니다.")
else:
    # 캐싱을 아예 안 썼다면 냈을 비용 (전량 uncached 입력가)
    nocache_input = ptoks * PRICE_INPUT / 1_000_000
    write_extra = ptoks * PRICE_INPUT * (CACHE_WRITE_MULT - 1) / 1_000_000   # 쓰기로 더 내는 돈
    read_saved = ptoks * (PRICE_INPUT - PRICE_CACHED_INPUT) / 1_000_000      # 읽기 1회로 아끼는 돈
    breakeven = write_extra / read_saved if read_saved else float("inf")

    print(f"입력 {ptoks:,} 토큰 기준 (출력 제외)")
    print(f"  캐싱 미사용 입력 비용      : ${nocache_input:.5f}")
    print(f"  캐시 쓰기로 더 내는 돈     : ${write_extra:.5f}  (1.25x 의 초과분 0.25x)")
    print(f"  캐시 읽기 1회로 아끼는 돈  : ${read_saved:.5f}  (0.9x 할인)")
    print(f"\n  → 손익분기: 쓰기 후 읽기를 {breakeven:.2f}회 이상 하면 이득")
    print(f"     즉 같은 프리픽스를 총 {1 + breakeven:.2f}회 이상 사용해야 합니다.")
    print("\n  ⚠️ 단발 요청(재사용 0회)이라면 5.6 은 캐싱 때문에 오히려 1.25배 비쌉니다.")
    print("     5.5 이하는 쓰기가 무료라 이 리스크가 없었습니다 — 세대 전환 시 주의할 지점입니다.")

    n_reuse = [1, 2, 5, 10, 50]
    print(f"\n  [재사용 횟수별 총 입력 비용 — 캐싱 사용 vs 미사용]")
    print(f"  {'재사용':>6} | {'캐싱 사용':>12} | {'캐싱 미사용':>12} | {'절감':>8}")
    for n in n_reuse:
        with_cache = (ptoks * PRICE_INPUT * CACHE_WRITE_MULT + ptoks * PRICE_CACHED_INPUT * n) / 1_000_000
        without = ptoks * PRICE_INPUT * (1 + n) / 1_000_000
        print(f"  {n:>6} | ${with_cache:>11.5f} | ${without:>11.5f} | {(1 - with_cache / without) * 100:>7.1f}%")

In [ ]:
# === 캐시 입자도(granularity) 관찰 ===
# 5.6 은 5.5 이하(128토큰 블록 절삭)보다 입자도가 세밀합니다. 실제로 얼마나 남는지 확인합니다.
print(f"{'call':>5} | {'prompt':>8} | {'cached':>8} | {'write':>8} | {'미캐시 꼬리':>10} | {'적중률':>7}")
print("-" * 64)
for i, r in enumerate(hit, 1):
    tail = (r["prompt_tokens"] or 0) - r["cached"]
    rate = r["cached"] / r["prompt_tokens"] if r["prompt_tokens"] else 0
    print(f"{i:>5} | {r['prompt_tokens']:>8,} | {r['cached']:>8,} | {r['write']:>8,} | {tail:>10,} | {rate:>6.2%}")

tails = [(r["prompt_tokens"] or 0) - r["cached"] for r in hit if r["cached"]]
if tails:
    print(f"\n미캐시 꼬리 중앙값: {statistics.median(tails):,.0f} 토큰")
    for block in (128, 256, 512, 1024):
        if any(c["cached"] % block == 0 for c in hit if c["cached"]):
            print(f"  → cached_tokens 가 {block} 의 배수 — 캐시 블록 단위가 {block} 토큰으로 추정됩니다.")
            break
    else:
        print("  → 흔한 블록 크기의 배수가 아닙니다 (거의 전량 캐싱 = 세밀한 입자도).")
        print("     같은 조건에서 gpt-5.4-mini 는 128 의 배수로 절삭됩니다 — gpt54_mini 폴더와 비교해 보세요.")

In [ ]:
# === 막대그래프 (matplotlib 있으면) ===
try:
    import matplotlib.pyplot as plt

    labels = ["TTFT", "total"]
    miss_ms = [summary["ttft"]["miss"]["mean"] * 1000, summary["total"]["miss"]["mean"] * 1000]
    hit_ms = [summary["ttft"]["hit"]["mean"] * 1000, summary["total"]["hit"]["mean"] * 1000]
    xs, w = range(len(labels)), 0.35
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar([x - w / 2 for x in xs], miss_ms, w, label="MISS (cold, cache write)")
    ax.bar([x + w / 2 for x in xs], hit_ms, w, label="HIT (warm, cache read)")
    ax.set_ylabel("ms")
    ax.set_xticks(list(xs))
    ax.set_xticklabels(labels)
    ax.set_title(f"{MODEL} KV cache: cold vs warm (~{ptoks:,} tok)")
    ax.legend()
    for i, (mv, hv) in enumerate(zip(miss_ms, hit_ms)):
        ax.text(i - w / 2, mv, f"{mv:.0f}", ha="center", va="bottom", fontsize=8)
        ax.text(i + w / 2, hv, f"{hv:.0f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("시각화 건너뜀:", e)

In [ ]:
# === 결과 저장 (4개 노트북 공통 스키마 → 교차 비교용) ===
result = {
    "provider": "openai",
    "model": MODEL,
    "cache_mode": "automatic-prefix-with-write-billing",   # 5.6 세대: 쓰기/읽기 분리
    "target_tokens": TARGET_TOKENS,
    "prompt_tokens": ptoks,
    "n_repeat": N_REPEAT,
    "ttft_miss_ms": summary["ttft"]["miss"]["mean"] * 1000,
    "ttft_hit_ms": summary["ttft"]["hit"]["mean"] * 1000,
    "ttft_improve_pct": summary["ttft"]["improve_pct"],
    "total_miss_ms": summary["total"]["miss"]["mean"] * 1000,
    "total_hit_ms": summary["total"]["hit"]["mean"] * 1000,
    "total_improve_pct": summary["total"]["improve_pct"],
    "cached_tokens_hit": avg_cached_hit,
    "cached_tokens_miss": avg_cached_miss,
    "hit_rate": avg_cached_hit / ptoks if ptoks else 0,
    "cost_miss_usd": miss_cost,
    "cost_hit_usd": hit_cost,
    "cost_save_pct": save_pct,
    # 5.6 세대 고유 필드
    "cache_write_tokens_hit": avg_write_hit,
    "cache_write_tokens_miss": avg_write_miss,
    "cache_write_mult": CACHE_WRITE_MULT,
}
out = pathlib.Path("results.json")
out.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"저장: {out.resolve()}")
print(json.dumps(result, indent=2, ensure_ascii=False))

## 결과 해석 가이드

- **TTFT 개선율 > 전체 latency 개선율** 이 정상입니다. 캐싱은 입력을 처리하는 **prefill 구간만** 단축하고 디코딩 구간은 그대로입니다.
- **`cached_tokens` 가 진짜 지표입니다.** 시간 수치는 서버 부하로 흔들리지만 토큰 usage 는 흔들리지 않습니다.
- **`cache_write_tokens` 를 같이 보세요.** MISS 호출마다 write 가 잡히면 그만큼 1.25배를 내고 있다는 뜻입니다. 프리픽스에 타임스탬프·UUID 가 섞여 매번 캐시를 새로 쓰는 상황이 5.6 에서는 **직접적인 비용 손실**로 이어집니다.
- **cached_tokens 가 0 이면**: ① 워밍업 직후 미적재 ② evict ③ 프리픽스 1024 토큰 미만 — MISS→HIT 셀을 **연속 재실행**하세요.

### 이 규모(20K)에서 실제로 관측된 것

20,000 토큰 · 반복 3회로 실행했을 때 (⚠️ **아래 비용 수치는 쓰기 회계를 반영하기 전 구 단가 기준**입니다 — 재실행하면 갱신됩니다):

| 지표 | 값 |
|---|---|
| 캐시 적중률 | **99.98%** (16,289 / 16,292 토큰) |
| TTFT 개선 | +7.3% (미미) |
| 전체 latency 개선 | −1.5% (오히려 느림) |

**적중률은 최고인데 시간 개선이 거의 없습니다.** 20K 규모에서는 prefill 자체가 수백 ms 라 네트워크·서버 부하 변동에 묻히기 때문입니다. 같은 조건에서 `gpt-5.4-mini` 는 적중률이 더 낮은데도(98.99%) TTFT 가 44% 빨라졌습니다 — **적중률과 시간 이득은 별개**입니다.

시간 이득을 제대로 보려면:
1. **`TARGET_TOKENS` 를 100K~200K 로 올리기** — prefill 이 전체 latency 를 지배하게 만듭니다. (가장 확실)
2. **`N_REPEAT` 를 10 이상으로 올리기** — 노이즈를 평균으로 눌러 없앱니다.

**교훈**: 캐싱이 작동하는지 판정할 때는 시간이 아니라 **토큰 usage 필드**를 보세요.
